# QUOA-Starter
This is the initial vibe-coded process from Claude.

## Pipeline
Your pipeline, step by step:
1. **Cost table → QUBO.** A QUBO is a formula that scores every possible answer. Low score means a good plan. It adds the delivery costs, plus a penalty (1.5) when a plan breaks the rules: a bot taking two orders, or an order left unserved. Each (bot, order) pair is one yes/no variable, which is one qubit. 2 bots × 2 orders = 4 qubits, and 3 bots × 3 orders = 9.
2. **QUBO → quantum circuit.** Qiskit turns the formula into a circuit with two dials, gamma and beta. The circuit makes a spread of possible answers. The right dial settings make good answers more likely.
3. **Tune the dials.** A classical optimizer (here COBYLA) tries gamma and beta over and over, checking the average score each time. This loop is why starting values matter. Better starting angles mean fewer iterations, which is what C’s ML model predicts.
4. **Measure and decode.** Run the circuit many times and take the most common bitstring, e.g. 1001. Read it as “bot 0 takes order 0, bot 1 takes order 1” and compute the cost.
5. **Hand off:** {bot_id: order_id} plus total cost.

The script does steps 1–4 on a fake 2×2 problem and compares the result to brute force.

In [42]:
!pip install qiskit qiskit-aer scipy numpy -q

In [43]:
# Generated by Claude
"""
Day-1 starter for Person B (quantum/qaoa_starter.py).

Goal for today: run QAOA alone on a tiny FAKE problem (2 bots x 2 orders = 4 qubits)
and check it against brute force. No simulator/ML code needed yet.

Install first:   pip install qiskit qiskit-aer scipy numpy
NOTE: written without being able to run it (no PyPI access in my sandbox).
If an import or API call errors on your Qiskit version, paste the error back.

TODO tomorrow: replace toy_qubo() with the team's build_qubo from campus_data.py
(costs scaled 0-1, penalty 1.5). Do NOT keep this toy version.
"""
import itertools
import numpy as np
from scipy.optimize import minimize
from qiskit.circuit.library import QAOAAnsatz
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

PENALTY = 1.5  # must match the team's build_qubo


# ---------- 1. Problem -> QUBO (toy stand-in for the shared build_qubo) ----------
def toy_qubo(cost, penalty=PENALTY):
    """cost[b][o] = cost (0-1) of bot b delivering order o.
    Qubit index = b * n_orders + o. x=1 means "bot b takes order o".
    Energy = sum cost*x + penalty*(each bot takes exactly 1)^2 + penalty*(each order taken exactly once)^2
    Returns (Q, offset) with energy(x) = x^T Q x + offset.
    """
    cost = np.asarray(cost, float)
    nb, no = cost.shape
    n = nb * no
    Q = np.zeros((n, n))
    offset = 0.0
    idx = lambda b, o: b * no + o
    for b in range(nb):
        for o in range(no):
            Q[idx(b, o), idx(b, o)] += cost[b, o]

    def exactly_one(vars_):  # penalty * (sum x - 1)^2, using x^2 = x
        nonlocal offset
        offset += penalty
        for i in vars_:
            Q[i, i] += -penalty
        for i, j in itertools.combinations(vars_, 2):
            Q[i, j] += 2 * penalty

    for b in range(nb):
        exactly_one([idx(b, o) for o in range(no)])
    for o in range(no):
        exactly_one([idx(b, o) for b in range(nb)])
    return Q, offset




In [44]:
# ---------- 2. QUBO -> quantum "cost Hamiltonian" ----------
def qubo_to_pauli(Q, offset=0.0):
    """Substitute x = (1 - Z)/2 to get a sum of Z terms Qiskit understands."""
    n = Q.shape[0]
    const = offset
    lin = np.zeros(n)
    terms = []
    for i in range(n):
        const += Q[i, i] / 2
        lin[i] -= Q[i, i] / 2
        for j in range(i + 1, n):
            q = Q[i, j]
            if q == 0:
                continue
            const += q / 4
            lin[i] -= q / 4
            lin[j] -= q / 4
            terms.append((i, j, q / 4))
    pauli = []
    for i in range(n):
        if lin[i] != 0:
            s = ["I"] * n; s[n - 1 - i] = "Z"      # Qiskit: qubit 0 is the rightmost char
            pauli.append(("".join(s), lin[i]))
    for i, j, c in terms:
        s = ["I"] * n; s[n - 1 - i] = "Z"; s[n - 1 - j] = "Z"
        pauli.append(("".join(s), c))
    return SparsePauliOp.from_list(pauli), const




In [45]:
# ---------- 3. Run QAOA ----------
def decode(bitstring, cost):
    nb, no = cost.shape
    grid = np.array([int(c) for c in bitstring[::-1]]).reshape(nb, no)  # grid[bot][order]
    valid = bool((grid.sum(axis=1) == 1).all() and (grid.sum(axis=0) == 1).all())
    assignment = {b: [int(o) for o in np.flatnonzero(grid[b])] for b in range(nb)}
    total = None
    if valid:
        assignment = {b: v[0] for b, v in assignment.items()}
        total = sum(cost[b, o] for b, o in assignment.items())
    return assignment, valid, total
    
def run_qaoa(cost, gamma, beta, reps=1, seed=0, penalty=PENALTY):
    """Return dict with assignment {bot: order}, total cost, iterations, valid flag.
    gamma, beta are the STARTING angles (this is where ML warm-start plugs in)."""
    cost = np.asarray(cost, float)
    nb, no = cost.shape
    Q, offset = toy_qubo(cost, penalty) #Change to build_qubo when given it
    H, const = qubo_to_pauli(Q, offset)

    circuit = QAOAAnsatz(cost_operator=H, reps=reps)
    circuit.measure_all()
    ansatz = QAOAAnsatz(cost_operator=H, reps=reps)  # no measurement, for the estimator
    estimator = StatevectorEstimator(seed=seed)
    sampler = StatevectorSampler(seed=seed)

    n_iter = {"n": 0}

    def energy(params):
        n_iter["n"] += 1
        pub = (ansatz, H, params)
        return float(estimator.run([pub]).result()[0].data.evs) + const

    # QAOAAnsatz parameter order: betas first, then gammas
    x0 = np.array([beta] * reps + [gamma] * reps, float)
    res = minimize(energy, x0, method="COBYLA", options={"maxiter": 200})

    bound = circuit.assign_parameters(res.x)
    counts = sampler.run([bound], shots=2048).result()[0].data.meas.get_counts()
    shots = sum(counts.values())
    top = sorted(counts.items(), key=lambda kv: kv[1], reverse=True)[:4]

    top_results = []
    for bitstring, n in top:
        assignment, valid, total = decode(bitstring, cost)
        top_results.append({"bits": bitstring, "prob": n / shots,
                            "assignment": assignment, "valid": valid, "cost": total})

    best = top_results[0]   # same winner as before
    # probability that a measurement gives the exact optimal plan:
    opt_assignment, _ = brute_force(cost)
    def is_opt(bs):
        a, ok, _ = decode(bs, cost)
        return ok and a == opt_assignment

    p_optimal = sum(n for bs, n in counts.items() if is_opt(bs)) / shots

    return {"assignment": best["assignment"], "cost": best["cost"], "valid": best["valid"],
            "iterations": n_iter["n"], "angles": res.x.tolist(),
            "top": top_results, "p_optimal": p_optimal}

In [46]:
# ---------- 4. Brute-force check (fine for 2-3 bots) ----------
def brute_force(cost):
    cost = np.asarray(cost, float)
    best = min(itertools.permutations(range(cost.shape[1]), cost.shape[0]),
               key=lambda p: sum(cost[b, o] for b, o in enumerate(p)))
    return {b: o for b, o in enumerate(best)}, sum(cost[b, o] for b, o in enumerate(best))

def benchmark(cost, reps=1, n_starts=20, seed=42):
    """Run QAOA from many random starting angles and summarize how it does."""
    rng = np.random.default_rng(seed)          # same seed -> same starting angles every call, good for comparison between layer numbers
    _, opt_cost = brute_force(cost)
    rows = []
    for _ in range(n_starts):
        g, b = rng.uniform(0, np.pi, size=2)
        out = run_qaoa(cost, gamma=g, beta=b, reps=reps)
        found_opt = out["valid"] and abs(out["cost"] - opt_cost) < 1e-9
        rows.append((found_opt, out["p_optimal"], out["iterations"]))
    ok, p, it = zip(*rows)
    return {"reps": reps,
            "winner_success_rate": float(np.mean(ok)),
            "mean_p_optimal": float(np.mean(p)),
            "mean_iterations": float(np.mean(it))}

if __name__ == "__main__":
    fake_cost = [[0.2, 0.9],
                 [0.7, 0.1]]
    print("exact answer :", brute_force(fake_cost))
    for g, b in [(0.5, 0.5), (1.0, 0.3), (0.2, 0.8)]:
        out = run_qaoa(fake_cost, gamma=g, beta=b, reps=2)
        print(f"start ({g}, {b}): p_optimal={out['p_optimal']:.2f}, iters={out['iterations']}")
        for t in out["top"]:
            print("   ", t["bits"], f"{t['prob']:.2f}", t["assignment"], t["cost"])

    for reps in (1, 2):
        print(benchmark(fake_cost, reps=reps))

exact answer : ({0: 0, 1: 1}, np.float64(0.30000000000000004))
start (0.5, 0.5): p_optimal=0.41, iters=80
    1001 0.41 {0: 0, 1: 1} 0.30000000000000004
    0110 0.38 {0: 1, 1: 0} 1.6
    1011 0.04 {0: [0, 1], 1: [1]} None
    1101 0.03 {0: [0], 1: [0, 1]} None
start (1.0, 0.3): p_optimal=0.43, iters=200
    1001 0.43 {0: 0, 1: 1} 0.30000000000000004
    0110 0.40 {0: 1, 1: 0} 1.6
    1111 0.04 {0: [0, 1], 1: [0, 1]} None
    0000 0.03 {0: [], 1: []} None
start (0.2, 0.8): p_optimal=0.43, iters=200
    1001 0.43 {0: 0, 1: 1} 0.30000000000000004
    0110 0.40 {0: 1, 1: 0} 1.6
    1111 0.04 {0: [0, 1], 1: [0, 1]} None
    0000 0.03 {0: [], 1: []} None
{'reps': 1, 'winner_success_rate': 0.3, 'mean_p_optimal': 0.1928466796875, 'mean_iterations': 40.0}
{'reps': 2, 'winner_success_rate': 0.9, 'mean_p_optimal': 0.4150634765625, 'mean_iterations': 118.85}
